**1. Install & Imports**

In [ ]:
!pip install transformers datasets==2.16.1 evaluate seqeval

import numpy as np
from datasets import (
    load_dataset
)
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from evaluate import load

**2. Load Dataset**

In [ ]:
dataset = load_dataset("conll2003", revision="main")

print(dataset)
print(dataset["train"][0])

**3. Labels**

In [ ]:
label_list = dataset["train"].features["ner_tags"].feature.names
num_labels = len(label_list)

print(label_list)

**4. Tokenizer**

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

**5. Tokenization + Label Alignment**

In [ ]:
def tokenize_and_align_labels(example):
    tokenized = tokenizer(
        example["tokens"],
        truncation=True,
        padding="max_length",
        is_split_into_words=True
    )

    labels = []
    word_ids = tokenized.word_ids()
    prev_word = None

    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)
        elif word_idx != prev_word:
            labels.append(example["ner_tags"][word_idx])
        else:
            labels.append(-100)

        prev_word = word_idx

    tokenized["labels"] = labels
    return tokenized

**6. Apply Preprocessing**

In [ ]:
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=False)

**7. Model**

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels,
    id2label={i: l for i, l in enumerate(label_list)},
    label2id={l: i for i, l in enumerate(label_list)}
)

**6. Apply Preprocessing**

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    logging_strategy="epoch",
    save_strategy="no"
)

**9. Data Collator**

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer)

**10. Evaluation**

In [ ]:
metric = load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_preds = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labels = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_preds, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"]
    }

**11. Trainer**

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

**12. Train**

In [ ]:
trainer.train()

**13.Evaluate**

In [ ]:
trainer.evaluate()

**14. Inference**

In [ ]:
import torch

sentence = "John works at Google in California"

inputs = tokenizer(
    sentence.split(),
    return_tensors="pt",
    is_split_into_words=True
)

outputs = model(**inputs).logits
predictions = torch.argmax(outputs, dim=2)

tokens = sentence.split()

for token, pred in zip(tokens, predictions[0]):
    print(token, "->", label_list[pred.item()])